In [1]:
import pandas as pd

In [2]:
df_index = pd.read_excel(r'shared_data_read_only/District_and_School_Performance_Index_Ranking.xlsx', sheet_name='Building PI Rankings')
df_value = pd.read_excel(r'shared_data_read_only/District_Value_Added_Ranking.xlsx', sheet_name='Value-Added Rankings 2023')
#df_lookup = pd.read_csv(r'shared_data_read_only/NCES_School_Lookup.csv')

In [3]:
df_merge = df_index.merge(df_value, left_on='LEA IRN', right_on='District IRN')
df_merge = df_merge[df_merge['2023 PI for Ranking'] != 'NC']

In [4]:
df_new = df_merge.groupby('ODE Designated County_x').size().reset_index(name='counts').sort_values(by='counts')
df_new = df_new[df_new['counts']>20]

In [5]:
counties = []
for i in range(len(df_new)):
    counties.append(df_new.iloc[i,0])

In [6]:
df_merge['2023 PI for Ranking'] = df_merge["2023 PI for Ranking"].astype(float)
grouped_county = df_merge.groupby('ODE Designated County_x')
grouped_dict_variance_county = grouped_county['2023 PI for Ranking'].var().to_dict()
grouped_dict_counts_county = grouped_county.size().to_dict()

In [7]:
df_merge['Variance'] = None
for index, row in df_merge.iterrows():
    currCounty = row['ODE Designated County_x']
    if grouped_dict_counts_county[currCounty] >= 20:
        df_merge.at[index, 'Variance'] = grouped_dict_variance_county[currCounty]
    else:
        df_merge.at[index, 'Variance'] = 1000

In [8]:
df_merge[df_merge['Variance']==1000]

,2023 PI Ranking,LEA Name,Building Name,2023 PI for Ranking,2023 PI Proxy Descriptor,2023 Achievement Rating,ODE Designated County_x,LEA IRN,Building IRN,Building Org Type,...,ODE Designated County_y,District Org Type,District Org Status,2023 Grade Span_y,2023 Value-Added Star Rating,2023 Value-Added Gain Index,2023 Value-Added Effect Size,2023 Value-Added Index Ranking,2023 Value-Added Effect Size Ranking,Variance
33,12,St Henry Consolidated Local,St Henry Elementary School,112.333,REAL PI,5 Stars,Mercer,48587,35634,Public School,...,Mercer,Public District,Open,K-12,1 Star,-6.55,-0.13,775,748,1000
34,92,St Henry Consolidated Local,St Henry Middle School,106.872,REAL PI,5 Stars,Mercer,48587,35642,Public School,...,Mercer,Public District,Open,K-12,1 Star,-6.55,-0.13,775,748,1000
35,291,St Henry Consolidated Local,St Henry High School,102.633,REAL PI,5 Stars,Mercer,48587,35659,Public School,...,Mercer,Public District,Open,K-12,1 Star,-6.55,-0.13,775,748,1000
39,116,Dublin City,Glacier Ridge Elementary,106.190,REAL PI,5 Stars,Union,47027,8257,Public School,...,Franklin,Public District,Open,K-12,4 Stars,15.84,0.09,11,145,1000
43,247,Dublin City,Eversole Run Middle School,103.431,REAL PI,5 Stars,Union,47027,19393,Public School,...,Franklin,Public District,Open,K-12,4 Stars,15.84,0.09,11,145,1000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3046,2562,Trimble Local,Trimble Junior High,67.210,REAL PI,2 Stars,Athens,45922,65318,Public School,...,Athens,Public District,Open,K-12,3 Stars,-0.29,0.00,434,398,1000
3047,2600,Trimble Local,Trimble High School,66.093,REAL PI,2 Stars,Athens,45922,13755,Public School,...,Athens,Public District,Open,K-12,3 Stars,-0.29,0.00,434,398,1000
3087,2604,Lakeland Academy Community School,Lakeland Academy Community School,66.000,REAL PI,2 Stars,Harrison,11511,11511,Community School,...,Harrison,Community School,Open,"K-12,P",3 Stars,-1.76,-0.23,541,821,1000
3179,2925,Bridges Community Academy dba Bridges Preparat...,Bridges Community Academy dba Bridges Preparat...,54.153,REAL PI,1 Star,Seneca,311,311,Community School,...,Seneca,Community School,Open,K-12,1 Star,-3.09,-0.24,629,823,1000


In [10]:
df_merge.to_excel('variance.xlsx', columns = ['ODE Designated County_x', 'Variance'])